# Search-Based Intelligence (SBI) — Experiment Notebook

**Goal:** Prove that a 100M transformer + memory system outperforms the same 100M transformer without memory on bAbI reasoning tasks.

**Tasks:**
- bAbI Task 1 — single supporting fact
- bAbI Task 2 — two supporting facts ← memory helps here
- bAbI Task 3 — three supporting facts ← memory helps most here

**Baseline:** Memory Networks (Weston et al. 2015) published results for comparison.


In [ ]:
# ── Setup ───────────────────────────────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/sbi.git
%cd sbi
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print(f'PyTorch: {torch.__version__}')

## 1. Sanity Check — all components load correctly

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
from sbi.system import SBISystem
from sbi.core.config import SBIConfig, ReasoningConfig, MemoryConfig

cfg = SBIConfig(
    reasoning=ReasoningConfig(vocab_size=300, n_layers=4, n_heads=4, d_model=256, d_ff=1024, hidden_dim=256),
    memory=MemoryConfig(fingerprint_dim=64),
)
system = SBISystem(cfg)
print(f'Parameters: {system.num_parameters():,}')

dummy = torch.randint(0, 300, (2, 32))
logits, fp = system(dummy)
print(f'Logits: {logits.shape}, Fingerprint: {fp.shape}')
print('Sanity check PASSED')

## 2. Inspect bAbI Data

In [ ]:
from data.babi.loader import load_babi_tasks, flatten_example

examples = load_babi_tasks(task_ids=[1, 2, 3], split='train', size_per_task=3)

for ex in examples:
    inp, ans = flatten_example(ex)
    print(f'[Task {ex["task_id"]}]')
    print(f'  Input:  {inp}')
    print(f'  Answer: {ans}')
    print()

## 3. Train Baseline (no memory)

In [ ]:
!python training/train_baseline.py --config configs/baseline.yaml

## 4. Train SBI System (with memory)

In [ ]:
!python training/train_sbi.py --config configs/sbi_small.yaml

## 5. Compare Results

In [ ]:
!python training/evaluate.py

from IPython.display import Image
Image('experiments/results/comparison.png')

## 6. Memory System Analysis
How much did the system actually use its memory?

In [ ]:
import sys, torch, matplotlib.pyplot as plt
sys.path.insert(0, 'src')

from sbi.system import SBISystem
from sbi.core.config import SBIConfig, ReasoningConfig, MemoryConfig
from data.babi.dataset import BabiDataset

train_ds = BabiDataset(task_ids=[1,2,3], split='train', max_seq_len=256)
vocab_size = train_ds.vocab_size

sbi_config = SBIConfig(
    reasoning=ReasoningConfig(vocab_size=vocab_size, hidden_dim=768),
    memory=MemoryConfig(),
)
sbi = SBISystem(sbi_config)
ckpt = torch.load('experiments/checkpoints/sbi_best.pt', map_location='cpu')
sbi.load_state_dict(ckpt['model_state'])

stats = sbi.memory_stats()
print(f"Memory entries : {stats['memory_size']}")
print(f"Hebbian edges  : {stats['graph_edges']}")
print(f"Training steps : {stats['step']}")

if sbi.episodic_memory.entries:
    confidences = [e.confidence for e in sbi.episodic_memory.entries]
    usage_counts = [e.usage_count for e in sbi.episodic_memory.entries]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(confidences, bins=20, color='steelblue', edgecolor='white')
    axes[0].set_title('Confidence Distribution')
    axes[0].set_xlabel('Confidence')

    axes[1].hist(usage_counts, bins=20, color='darkorange', edgecolor='white')
    axes[1].set_title('Memory Usage Count')
    axes[1].set_xlabel('Times Retrieved')

    plt.tight_layout()
    plt.savefig('experiments/results/memory_analysis.png', dpi=150)
    plt.show()
else:
    print('No memory entries found — memory threshold may be too high.')

## 7. Reference — Published Memory Networks Results

| Task | MemNN (Weston 2015) | End-to-End MemNN |
|------|--------------------|-----------------|
| Task 1 | 100% | 99.9% |
| Task 2 | 100% | 98.3% |
| Task 3 | 74%  | 62.7% |

Our goal: SBI should close the gap on Tasks 2 and 3 compared to the baseline transformer.